In [83]:
from langchain_core.tools  import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage,ToolMessage
import requests, json


In [ ]:
@tool
def getcurrencyrates(base_currency: str, desired_currency:str)->float:
    """Currency rates provider, provides the latest currency exchange rates"""

    url=f"https://v6.exchangerate-api.com/v6/your_api_key/pair/{base_currency.upper()}/{desired_currency.upper()}"

    response=requests.get(url)
    return response.json()



In [57]:

@tool
def convert(base_currency : int, conversion_rate:float)->float:
    """This tool is mainly used to convert the data from the given base currency to the desired currency"""

    return base_currency*conversion_rate

In [92]:
tool_map={
    "getcurrencyrates":getcurrencyrates,
    "convert":convert
}

tool_map["getcurrencyrates"]

StructuredTool(name='getcurrencyrates', description='Currency rates provider, provides the latest currency exchange rates', args_schema=<class 'langchain_core.utils.pydantic.getcurrencyrates'>, func=<function getcurrencyrates at 0x7e760eb5fa60>)

In [81]:
llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
model_with_tool=llm.bind_tools(list(tool_map.values()))

In [94]:
def calculatecurrencyapp(query:str):
    message=[HumanMessage(content=query)]

    max_iterations=5

    for _ in range(max_iterations):
        response=model_with_tool.invoke(message)
        message.append(response)

        if not response.tool_calls:
            print(response.text)
            return response
        
        for tool_call in response.tool_calls:
            tool_name=tool_call['name']
            tool_id=tool_call['id']
            tool_args=tool_call['args']

            selected_tool=tool_map[tool_name]
            tool_output=selected_tool.invoke(tool_args)

            tool_message=ToolMessage(
                content=str(tool_output),
                name=tool_name,
                tool_call_id=tool_id 
            )
            message.append(tool_message)
            


In [97]:
calculatecurrencyapp(query="What is the current rate of inr in us dollars and uk")

The current exchange rates for the Indian Rupee (INR) are as follows:

*   **1 INR = 0.01044 USD** (US Dollars)
*   **1 INR = 0.007788 GBP** (British Pound Sterling)

*Please note: These rates are based on the latest available data as of May 29, 2026, and may fluctuate.*


AIMessage(content=[{'type': 'text', 'text': 'The current exchange rates for the Indian Rupee (INR) are as follows:\n\n*   **1 INR = 0.01044 USD** (US Dollars)\n*   **1 INR = 0.007788 GBP** (British Pound Sterling)\n\n*Please note: These rates are based on the latest available data as of May 29, 2026, and may fluctuate.*', 'extras': {'signature': 'EjQKMgEMOdbHraQr8a61vor3Q/k7bFn0ZgvNHPpQNHrRcf0NYFgh34fTD3rBSG/dlJQe1Rix'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7415-593a-71f1-a147-47268876970b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 611, 'output_tokens': 91, 'total_tokens': 702, 'input_token_details': {'cache_read': 0}})